- This setup is for baseline model investigations
- Default hyperparameters are used, this would be tuned later if selected


In [ ]:
import pandas as pd
import numpy as np
from sklearn.ensemble import RandomForestRegressor
from sklearn import metrics, model_selection

In [10]:
train = pd.read_csv("../artifacts/feature_engineering/features_train.csv")
test = pd.read_csv("../artifacts/feature_engineering/features_test.csv")

target = "log_price"

In [11]:
X_train = train.drop(columns=target)
y_train = train[target]

X_test = test.drop(columns=target)
y_test = test[target]

In [12]:
model = RandomForestRegressor()
model = model.fit(X_train, y_train)

In [13]:
def compute_metrics(model, x, y, cv=5):
    preds = model.predict(x)
    score = model.score(x, y)

    scores_cvs = model_selection.cross_val_score(model, x, y, scoring="r2", cv=cv)

    return pd.DataFrame(
        [
            {
                "R2": round(score, 3),
                "mse": round(metrics.mean_squared_error(y, preds), 3),
                "rmse": round(np.sqrt(metrics.mean_squared_error(y, preds)), 3),
                "mae": round(metrics.mean_absolute_error(y, preds), 3),
                "adjusted_r2": round(
                    1 - (1 - score) * (len(y) - 1) / (len(y) - x.shape[1] - 1), 3
                ),
                "cv_score": round(scores_cvs.mean() * 100, 2),
            }
        ]
    )


In [ ]:
val_metrics = compute_metrics(model, X_test, y_test)
val_metrics

In [16]:
variables = abs(model.feature_importances_)
coef_df = pd.DataFrame(
    {
        "Variable": train.drop(["log_price"], axis=1).columns,
        "Value": variables,
    }
)
n = 20
sorted_df = (
    coef_df.sort_values(by="Value", ascending=False).head(n).sort_values(by="Value")
)
sorted_df


,Variable,Value
10,is_elite_tier,0.002665
11,class_pi,0.003737
14,loc_trust_score,0.004033
9,dist_to_wealth_hub,0.005333
17,dist_to_cbd,0.005366
19,min_dist_to_hub,0.006482
13,loc_std_dev,0.007548
18,dist_to_east_legon_center,0.008613
8,rel_size,0.008859
6,bath_per_bed,0.009693
